# 🖥️ Computer Vision Lab 6-1 — Perceptrons, PyTorch & the Training Loop

**BITS F459 — Computer Vision**

Runnable companion to [`labsheet6-1.md`](https://github.com/Aritra/Computer_Vision/blob/main/labsheets/labsheet6-1.md). The labsheet carries the
full explanations; this notebook carries the code, in order, ready to run.

**Lab 6 comes in two parts.** This notebook (6-1) teaches the PyTorch machinery on
**IRIS** — small enough that nothing hides behind long runtimes.
[Lab 6-2](https://github.com/Aritra/Computer_Vision/blob/main/labsheets/labsheet6-2.md) then points the same machinery at **images**.

Sections 3–5 use `%%writefile` to create `dataset.py`, `model.py` and `train.py` as
**real files on disk**, which later cells import as modules. You end up with a genuine
project directory you can also run from a terminal with `python run_lab6.py`.

> ⚠️ **Before running anything, check your kernel.** This notebook needs the
> **Python (cv-dl)** kernel. If the first cell raises `ModuleNotFoundError: No module
> named 'torch'`, you are on the wrong kernel — see Lab 6-1 Section 0, Step 3.
> **Lab 6-2 reuses the three files this notebook writes**, so keep this folder.

---
## 0️⃣ Setup

Environment and kernel setup is in **Lab 6-1 Section 0**. In short:

```bash
conda create -n cv-dl python=3.10 && conda activate cv-dl
pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu
pip install scikit-learn scikit-image matplotlib tensorboard opencv-python jupyter ipykernel
python -m ipykernel install --user --name cv-dl --display-name "Python (cv-dl)"
```

Then select **Python (cv-dl)** as this notebook's kernel. The `sys.executable` line below
is what proves you actually did — it must contain `cv-dl`.

In [ ]:
import sys
import torch, torchvision, skimage

print("python:     ", sys.executable)        # <- must contain 'cv-dl'
print("torch:      ", torch.__version__)
print("torchvision:", torchvision.__version__)
print("skimage:    ", skimage.__version__)
print("CUDA available:", torch.cuda.is_available())  # False is fine for this lab

---
## 1️⃣ From Lab 4's SVM to a Perceptron

In Lab 4 you trained `SGDClassifier(loss="hinge", ...)`. That object holds one weight
matrix and one bias, predicts with `W·x + b`, and updates `W` by gradient descent.
**That is a single-layer perceptron** — you have already built one.

A single layer draws **linear** boundaries. Stacking layers only helps if you put a
**non-linearity** between them: without one, `W2·(W1·x + b1) + b2` collapses
algebraically into a single linear map.

---
## 2️⃣ PyTorch Essentials

Two dtype rules that cause most beginner errors:

- Features must be **`float32`** (NumPy defaults to `float64`)
- Labels must be **`int64`** (`torch.long`) — `CrossEntropyLoss` wants **class indices**

In [ ]:
import torch
import numpy as np

a = torch.tensor([1.0, 2.0, 3.0])
b = torch.zeros(2, 3)
c = torch.from_numpy(np.array([1.0, 2.0]))   # shares memory with the NumPy array

print(a.shape, a.dtype, a.device)
print(a.numpy())          # back to NumPy (CPU tensors only)

### Autograd: watch a gradient get computed

**The most important cell in this lab.** Build one neuron, run it forward, call
`.backward()`, then check PyTorch's answer against the chain rule done by hand:

```
dloss/dw = 2(y_pred - y_true) × y_pred(1 - y_pred) × x
```

In [ ]:
import torch

# One neuron: two inputs, two weights, one bias. requires_grad=True tells PyTorch
# to record every operation involving these tensors, so it can walk backward later.
x = torch.tensor([2.0, 3.0])
w = torch.tensor([0.5, -1.0], requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)

# ---- forward pass ----
z = (w * x).sum() + b      # z = w1*x1 + w2*x2 + b
y_pred = torch.sigmoid(z)
y_true = torch.tensor(1.0)
loss = (y_pred - y_true) ** 2

print(f"z       = {z.item():.6f}")
print(f"y_pred  = {y_pred.item():.6f}")
print(f"loss    = {loss.item():.6f}")

# ---- backward pass ----
loss.backward()            # populates w.grad and b.grad

print(f"w.grad  = {w.grad}")
print(f"b.grad  = {b.grad.item():.6f}")

In [ ]:
with torch.no_grad():
    manual_dz = 2 * (y_pred - y_true) * y_pred * (1 - y_pred)
    manual_dw = manual_dz * x
    manual_db = manual_dz

print(f"manual dw = {manual_dw}")
print(f"manual db = {manual_db.item():.6f}")

assert torch.allclose(w.grad, manual_dw), "gradients disagree!"
assert torch.allclose(b.grad, manual_db), "gradients disagree!"
print("autograd matches the hand-computed chain rule")

That `assert` passing is the point: `loss.backward()` is the chain rule applied
mechanically over the graph you just wrote.

### The update step

In [ ]:
LR = 0.1
with torch.no_grad():          # weight updates must not themselves be recorded
    w -= LR * w.grad
    b -= LR * b.grad

### The five-line ritual

```python
optimizer.zero_grad()              # 1. clear previous gradients
logits = model(features)           # 2. forward
loss = criterion(logits, labels)   # 3. loss
loss.backward()                    # 4. backward -> fills .grad
optimizer.step()                   # 5. update from those .grad values
```

> ⚠️ **Why `zero_grad()` exists.** PyTorch **accumulates** gradients rather than
> overwriting them. Forget it and step 4 *adds* this batch's gradient to every previous
> batch's. The model still trains — badly, erratically — and **there is no error
> message.** The most common PyTorch bug, and it fails silently.

---
## 3️⃣ `dataset.py` — A `Dataset` from Scratch

`Dataset` answers two questions: how many samples (`__len__`) and give me sample `i`
(`__getitem__`). `DataLoader` handles batching and shuffling. That separation is why
you can swap IRIS for MNIST in Lab 6-2 without touching the training loop.

The next cell **writes `dataset.py` to disk**.

In [ ]:
%%writefile dataset.py
"""dataset.py - IRIS"""

import numpy as np
import torch
from torch.utils.data import Dataset
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


class IrisDataset(Dataset):
    """A Dataset over in-memory feature/label arrays.

    This is the minimal, complete Dataset: store the data, answer __len__,
    answer __getitem__. Everything else (batching, shuffling) is the
    DataLoader's job, not ours.
    """

    def __init__(self, features, labels):
        # float32 for the features (what nn.Linear expects) and int64 for the
        # labels (what nn.CrossEntropyLoss expects as CLASS INDICES).
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


def build_iris_datasets(val_size=0.2, test_size=0.2, random_state=42):
    """Returns (train_ds, val_ds, test_ds, scaler, class_names, n_features, n_classes).

    Note the three-way split: train to fit weights, val to choose the best
    checkpoint and to trigger early stopping, test touched ONCE at the end.
    """
    data = load_iris()
    X, y = data.data, data.target

    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=val_size + test_size, stratify=y, random_state=random_state
    )
    relative_test = test_size / (val_size + test_size)
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=relative_test, stratify=y_temp, random_state=random_state
    )

    # Fit the scaler on TRAIN ONLY, then apply it to val/test - exactly the same
    # rule as Lab 4's "fit the visual vocabulary on training descriptors only".
    # Fitting on all the data leaks val/test statistics into training.
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    X_test = scaler.transform(X_test)

    return (
        IrisDataset(X_train, y_train),
        IrisDataset(X_val, y_val),
        IrisDataset(X_test, y_test),
        scaler,
        # Plain python str, NOT numpy.str_ - these get written into the
        # checkpoint, and torch.load's weights_only=True default rejects
        # numpy scalar types. See the checkpointing warning in Section 6.
        [str(name) for name in data.target_names],
        X_train.shape[1],   # 4 features
        len(data.target_names),  # 3 classes
    )

> ⚠️ **The scaler is part of your model.** Lose the `StandardScaler` fitted alongside
> `best.pt` and the checkpoint is useless — raw centimetres through weights trained on
> standardized inputs gives confident nonsense. We save it **inside** the checkpoint.

---
## 4️⃣ `model.py` — Single-Layer and Multi-Layer Perceptrons

In [ ]:
%%writefile model.py
"""model.py"""

import torch.nn as nn


class SingleLayerPerceptron(nn.Module):
    """One nn.Linear, nothing else: logits = W·x + b.

    This is the direct PyTorch equivalent of Lab 4's SGDClassifier - one
    weight matrix, one bias, a linear decision boundary.
    """

    def __init__(self, in_features, num_classes):
        super().__init__()          # ALWAYS call this first in an nn.Module
        self.fc = nn.Linear(in_features, num_classes)

    def forward(self, x):
        # x: (batch_size, in_features)  ->  returns (batch_size, num_classes)
        # Returns RAW LOGITS. No softmax here - see the warning below.
        return self.fc(x)


class MLP(nn.Module):
    """A multi-layer perceptron: one or more hidden layers, each followed by
    a ReLU non-linearity and (optionally) dropout."""

    def __init__(self, in_features, num_classes, hidden_sizes=(128,), dropout=0.0):
        super().__init__()

        layers = []
        prev = in_features
        for hidden in hidden_sizes:
            layers.append(nn.Linear(prev, hidden))
            # Without a non-linearity here, stacked Linear layers collapse
            # algebraically into a single Linear layer (Section 1).
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = hidden

        # Output layer: no activation, no dropout - raw logits.
        layers.append(nn.Linear(prev, num_classes))

        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def build_model(name, in_features, num_classes, hidden_sizes=(128,), dropout=0.0):
    if name == "single":
        return SingleLayerPerceptron(in_features, num_classes)
    if name == "mlp":
        return MLP(in_features, num_classes, hidden_sizes, dropout)
    raise ValueError(f"Unknown model: {name}")

> ⚠️ **No softmax in `forward()`.** `nn.CrossEntropyLoss` applies `LogSoftmax`
> internally. Doing it yourself too gives a **double softmax**: squashed gradients,
> slower training, worse plateau — and **no error**. Return raw logits.

Check the parameter count yourself:

In [ ]:
from model import MLP

model = MLP(160, 10, hidden_sizes=(128,), dropout=0.2)
print(model)
print("trainable parameters:",
      sum(p.numel() for p in model.parameters() if p.requires_grad))

---
## 5️⃣ `train.py` — Loop, TensorBoard, Checkpointing, Early Stopping

One parameterized function, so every experiment in Lab 6-2 is a **config change**.

In [ ]:
%%writefile train.py
"""train.py - one parameterized training run, logged to TensorBoard and checkpointed."""

import os
import time
import copy
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter

from model import build_model


class OneHotMSELoss(nn.Module):
    """MSE against a one-hot target. Included deliberately as the WRONG tool
    for classification, so you can watch it underperform in Lab 6-2."""

    def __init__(self, num_classes):
        super().__init__()
        self.num_classes = num_classes
        self.mse = nn.MSELoss()

    def forward(self, logits, targets):
        one_hot = torch.zeros_like(logits)
        one_hot.scatter_(1, targets.view(-1, 1), 1.0)
        return self.mse(torch.softmax(logits, dim=1), one_hot)


def build_criterion(name, num_classes):
    if name == "cross_entropy":
        return nn.CrossEntropyLoss()      # expects RAW LOGITS + class indices
    if name == "hinge":
        return nn.MultiMarginLoss()       # multiclass hinge == Lab 4's linear SVM
    if name == "mse":
        return OneHotMSELoss(num_classes)
    raise ValueError(f"Unknown loss: {name}")


def build_optimizer(name, params, lr, weight_decay):
    if name == "sgd":
        return torch.optim.SGD(params, lr=lr, weight_decay=weight_decay)
    if name == "sgd_momentum":
        return torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=weight_decay)
    if name == "adam":
        return torch.optim.Adam(params, lr=lr, weight_decay=weight_decay)
    if name == "adamw":
        return torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    raise ValueError(f"Unknown optimizer: {name}")


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """Run the model over a loader and return (mean_loss, accuracy).

    model.eval() matters here: it disables dropout. @torch.no_grad() is a
    separate thing - it stops the graph being recorded (faster, less memory).
    """
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for features, labels in loader:
        features, labels = features.to(device), labels.to(device)
        logits = model(features)
        loss = criterion(logits, labels)
        total_loss += loss.item() * len(labels)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += len(labels)
    return total_loss / total, correct / total


def train(config, train_ds, val_ds, test_ds=None, scaler=None, class_names=None):
    """Train one configuration end to end. Returns a summary dict."""

    run_name = config["run_name"]
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    log_dir = os.path.join("runs", run_name)
    ckpt_dir = os.path.join("checkpoints", run_name)
    os.makedirs(ckpt_dir, exist_ok=True)
    writer = SummaryWriter(log_dir)

    train_loader = DataLoader(train_ds, batch_size=config["batch_size"], shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=config["batch_size"], shuffle=False)

    model = build_model(
        config["model"], config["in_features"], config["num_classes"],
        hidden_sizes=config.get("hidden_sizes", (128,)),
        dropout=config.get("dropout", 0.0),
    ).to(device)

    criterion = build_criterion(config["loss"], config["num_classes"])
    optimizer = build_optimizer(
        config["optimizer"], model.parameters(),
        config["lr"], config.get("weight_decay", 0.0),
    )

    # Log the model graph to TensorBoard's GRAPHS tab - a clickable picture of
    # the forward pass you traced by hand in Section 4.
    sample_batch = next(iter(train_loader))[0].to(device)
    writer.add_graph(model, sample_batch)

    best_val_loss = float("inf")
    best_val_acc = 0.0
    best_epoch = -1
    best_state = None
    patience_counter = 0
    start_time = time.perf_counter()

    for epoch in range(1, config["epochs"] + 1):
        # ---------------- train one epoch ----------------
        model.train()                      # dropout ON
        running_loss, correct, total = 0.0, 0, 0

        for features, labels in train_loader:
            features, labels = features.to(device), labels.to(device)

            # ---- the five-line ritual (Section 2) ----
            optimizer.zero_grad()
            logits = model(features)
            loss = criterion(logits, labels)

            # Optional L1 regularization. Unlike L2 (which lives in the
            # optimizer as weight_decay), L1 is not built in - you just add
            # the term to the loss, because a loss is only an expression
            # autograd knows how to differentiate.
            l1_lambda = config.get("l1_lambda", 0.0)
            if l1_lambda > 0:
                loss = loss + l1_lambda * sum(p.abs().sum() for p in model.parameters())

            loss.backward()
            optimizer.step()
            # ------------------------------------------

            running_loss += loss.item() * len(labels)
            correct += (logits.argmax(dim=1) == labels).sum().item()
            total += len(labels)

        train_loss = running_loss / total
        train_acc = correct / total
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)

        # ---------------- TensorBoard ----------------
        writer.add_scalars("loss", {"train": train_loss, "val": val_loss}, epoch)
        writer.add_scalars("accuracy", {"train": train_acc, "val": val_acc}, epoch)
        writer.add_scalar("generalization_gap", train_acc - val_acc, epoch)

        # Weight and GRADIENT histograms. .grad still holds the last batch's
        # gradients at this point - zero_grad() only runs at the top of the
        # next iteration. The gradient histograms are what let you SEE a
        # learning rate explode or a gradient vanish in Lab 6-2.
        for name, param in model.named_parameters():
            writer.add_histogram(f"weights/{name}", param, epoch)
            if param.grad is not None:
                writer.add_histogram(f"grads/{name}", param.grad, epoch)

        # ---------------- checkpointing ----------------
        checkpoint = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            # The optimizer state carries momentum / Adam moment estimates.
            # Resume without it and the loss visibly jumps as they rebuild.
            "optimizer_state_dict": optimizer.state_dict(),
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "config": config,
            "class_names": class_names,
            # Save the fitted scaler WITH the weights - see Section 3's warning.
            "scaler_mean": None if scaler is None else scaler.mean_.tolist(),
            "scaler_scale": None if scaler is None else scaler.scale_.tolist(),
        }
        torch.save(checkpoint, os.path.join(ckpt_dir, "last.pt"))

        # ---------------- best checkpoint + early stopping ----------------
        # NOTE: selection is on VALIDATION loss, never training loss. Val loss
        # and val accuracy can disagree - loss often starts rising from
        # overconfidence while accuracy still creeps up. Pick one, state which.
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_acc = val_acc      # accuracy AT the best-loss epoch
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
            torch.save(checkpoint, os.path.join(ckpt_dir, "best.pt"))
            marker = "  <- best"
        else:
            patience_counter += 1
            marker = ""

        print(f"[{run_name}] epoch {epoch:3d}/{config['epochs']}  "
              f"train_loss={train_loss:.4f} train_acc={train_acc:.4f}  "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}{marker}")

        if patience_counter >= config.get("patience", config["epochs"]):
            print(f"[{run_name}] early stopping at epoch {epoch} "
                  f"(no val improvement for {patience_counter} epochs). "
                  f"best.pt is from epoch {best_epoch}.")
            break

    elapsed = time.perf_counter() - start_time

    # ---------------- final test evaluation, using the BEST weights ----------------
    # Restore best.pt rather than keeping whatever the last epoch produced -
    # this is the entire reason for tracking a best checkpoint at all.
    test_loss = test_acc = None
    if test_ds is not None and best_state is not None:
        model.load_state_dict(best_state)
        test_loader = DataLoader(test_ds, batch_size=config["batch_size"], shuffle=False)
        test_loss, test_acc = evaluate(model, test_loader, criterion, device)
        print(f"[{run_name}] TEST (from best.pt, epoch {best_epoch}): "
              f"loss={test_loss:.4f} acc={test_acc:.4f}")

    # hparams tab: makes every run across BOTH halves of Lab 6 comparable
    # in one sortable table
    writer.add_hparams(
        {k: v for k, v in config.items() if isinstance(v, (int, float, str, bool))},
        {"hparam/best_val_loss": best_val_loss,
         "hparam/best_val_acc": best_val_acc,
         "hparam/test_acc": test_acc if test_acc is not None else 0.0},
    )
    writer.close()

    return {
        "run_name": run_name, "best_epoch": best_epoch,
        "best_val_loss": best_val_loss, "best_val_acc": best_val_acc,
        "test_acc": test_acc, "seconds": elapsed,
    }

> ⚠️ **Save a dict, never the model object.** `torch.save(model)` pickles the class
> definition and breaks the moment you edit `model.py`.
>
> ⚠️ **`torch.load` defaults to `weights_only=True`** (PyTorch 2.6+), so the config
> sticks to plain primitives and stores the scaler as two lists.

---
## 6️⃣ Start TensorBoard

Load it **before** training, then refresh as runs appear.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

### Then train: IRIS, single layer vs MLP

In [ ]:
"""run_lab6.py - minimal driver"""

from dataset import build_iris_datasets
from train import train

iris_train, iris_val, iris_test, scaler, iris_classes, n_feat, n_cls = build_iris_datasets()

base_iris = dict(
    in_features=n_feat, num_classes=n_cls, batch_size=16, epochs=100,
    loss="cross_entropy", optimizer="adam", lr=1e-2, weight_decay=0.0,
    patience=15,
)

train({**base_iris, "run_name": "iris_single", "model": "single"},
      iris_train, iris_val, iris_test, scaler, iris_classes)

train({**base_iris, "run_name": "iris_mlp", "model": "mlp", "hidden_sizes": (32,)},
      iris_train, iris_val, iris_test, scaler, iris_classes)

### 📝 Task

1. Report best epoch, best val loss and test accuracy for both runs.
2. **Single vs multi layer** — how much does the hidden layer buy you? `setosa` is
   linearly separable from the other two; `versicolor`/`virginica` overlap slightly.

   > ⚠️ **IRIS's test split is only 30 samples** — one sample is 3.3% of the score. The
   > MLP scoring *below* the single-layer model is noise, not evidence. Re-run with a
   > few `random_state` values. Knowing when a difference is too small to mean anything
   > is part of the skill.
3. **Watch the checkpointing work.** Which epoch did `best.pt` come from, versus which
   epoch did training stop at? That gap is what early stopping bought you.
4. Open **GRAPHS** and compare against the forward pass you traced by hand. Then open
   **HISTOGRAMS** and watch the weight distributions move.
5. Deliberately break it: comment out `optimizer.zero_grad()` and re-run. It still
   trains — note how the curve differs, and how easy that would be to miss. Put it back.

---
## 💾 What you just produced — **Lab 6-2 needs these files**

Running locally, everything is already saved on disk in this folder; there is nothing to
download or copy anywhere. Lab 6-2 imports `dataset.py`, `model.py` and `train.py`
directly, so keep them where they are.

In [ ]:
import os

print("files written by this notebook:")
for f in ["dataset.py", "model.py", "train.py"]:
    print(f"  {f:14s} {'OK' if os.path.exists(f) else 'MISSING'}")

for folder in ["runs", "checkpoints"]:
    if os.path.isdir(folder):
        print()
        print(f"{folder}/")
        for sub in sorted(os.listdir(folder)):
            print("  ", sub)

---
## 🧪 Student Task (Evaluative) — A Perceptron from Scratch in NumPy

Full description in [`labsheet6-1.md`](https://github.com/Aritra/Computer_Vision/blob/main/labsheets/labsheet6-1.md).

Implement a single-layer perceptron entirely in NumPy — **no `torch.nn`, no autograd,
no optimizer** — and train it on IRIS. You write the forward pass, the cross-entropy
loss, the analytic gradients, the update, and the mini-batch loop.

**The core deliverable:** build a `SingleLayerPerceptron` with the *same* initial
weights, run one forward/backward on the *same* batch, and **assert your hand-computed
`dW`/`db` match `model.fc.weight.grad`/`model.fc.bias.grad`**. Show that assert passing —
it is Section 2's one-neuron check at full layer scale.

Then extend to one hidden layer (`X → W1 → ReLU → W2 → softmax`) and verify those too.

*Think about:* what happens if you initialize all weights to zero? Try it. And why does
your NumPy version need no `zero_grad()`?

---

## ➡️ Next

Continue to **[Lab 6-2](https://github.com/Aritra/Computer_Vision/blob/main/labsheets/labsheet6-2.md)** — image features and training mechanics.

In [ ]:
# Starter scaffold for the NumPy perceptron.
import numpy as np
from dataset import build_iris_datasets

train_ds, val_ds, test_ds, scaler, class_names, n_features, n_classes = build_iris_datasets()
X_train = train_ds.features.numpy()
y_train = train_ds.labels.numpy()
print(X_train.shape, y_train.shape, n_features, n_classes)

def softmax(z):
    z = z - z.max(axis=1, keepdims=True)      # subtract max for numerical stability
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

# TODO: forward -> cross-entropy loss -> analytic dW, db -> update -> loop
# Then verify dW/db against PyTorch's .grad on identical initial weights.